# 🏥 Task 4: Disease Prediction from Medical Data
**CodeAlpha Machine Learning Internship**

### Objective
Predict the likelihood of diseases based on patient medical data.

### Approach
- SVM, Logistic Regression, Random Forest, XGBoost
- Datasets: Heart Disease, Diabetes, Breast Cancer (UCI ML Repository)
- Metrics: Accuracy, Precision, Recall, F1-Score, ROC-AUC

In [ ]:
# ─── Install required libraries (run once) ───
# !pip install scikit-learn pandas numpy matplotlib seaborn xgboost

In [ ]:
# ─── 1. Import Libraries ───
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

# XGBoost (optional — comment out if not installed)
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print('XGBoost available!')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed — will skip.')

print('Libraries imported!')

In [ ]:
# ─── 2. Load Datasets ───
# We'll demonstrate with all 3 datasets

# ── Dataset A: Breast Cancer (built-in sklearn) ──
bc = load_breast_cancer()
df_bc = pd.DataFrame(bc.data, columns=bc.feature_names)
df_bc['target'] = bc.target  # 0=malignant, 1=benign
print('Breast Cancer Dataset:')
print(f'  Shape: {df_bc.shape}')
print(f'  Classes: {bc.target_names}')
print(f"  Class counts:\n{df_bc['target'].value_counts()}\n")

# ── Dataset B: Heart Disease (UCI via URL) ──
try:
    url_heart = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heart_Disease_Data.csv'
    df_heart = pd.read_csv(url_heart)
    df_heart.columns = df_heart.columns.str.strip()
    # Target: 0=no disease, 1=disease
    df_heart['target'] = (df_heart.iloc[:, -1] > 0).astype(int)
    print('Heart Disease Dataset:')
    print(f'  Shape: {df_heart.shape}')
    HEART_AVAILABLE = True
except:
    print('Heart Disease: Could not load from URL (no internet). Skipping.')
    HEART_AVAILABLE = False

# ── Dataset C: Diabetes (Pima Indians) ──
try:
    url_diab = 'https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv'
    df_diab = pd.read_csv(url_diab)
    print('Diabetes Dataset:')
    print(f'  Shape: {df_diab.shape}')
    DIABETES_AVAILABLE = True
except:
    print('Diabetes: Could not load from URL. Skipping.')
    DIABETES_AVAILABLE = False

In [ ]:
# ─── 3. Exploratory Data Analysis ───
print('=== Breast Cancer Dataset Info ===')
print(df_bc.describe().T[['mean','std','min','max']].round(2))
print(f'\nMissing values: {df_bc.isnull().sum().sum()}')

In [ ]:
# ─── 4. Correlation Heatmap (top features) ───
top_features = df_bc.corr()['target'].abs().nlargest(11).index.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(df_bc[top_features].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0)
plt.title('Feature Correlation — Breast Cancer (Top 10)')
plt.tight_layout()
plt.savefig('disease_correlation.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 5. Preprocessing ───
def preprocess(df, target_col='target'):
    X = df.drop(target_col, axis=1)
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)
    return X_train, X_test, X_train_sc, X_test_sc, y_train, y_test, scaler

X_train, X_test, X_train_sc, X_test_sc, y_train, y_test, scaler = preprocess(df_bc)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# ─── 6. Train Multiple Models ───
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'SVM (RBF)':           (SVC(probability=True, random_state=42), True),
    'Random Forest':       (RandomForestClassifier(n_estimators=100, random_state=42), False),
}

if XGBOOST_AVAILABLE:
    models['XGBoost'] = (XGBClassifier(n_estimators=100, random_state=42,
                                        eval_metric='logloss', verbosity=0), False)

results = {}
for name, (model, needs_scaling) in models.items():
    X_tr = X_train_sc if needs_scaling else X_train
    X_te = X_test_sc  if needs_scaling else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    cv  = cross_val_score(model, X_tr, y_train, cv=5, scoring='accuracy').mean()
    results[name] = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
                     'accuracy': acc, 'roc_auc': auc, 'cv_mean': cv}
    print(f'\n{name}:')
    print(f'  Accuracy  = {acc:.4f}')
    print(f'  ROC-AUC   = {auc:.4f}')
    print(f'  CV (5-fold) = {cv:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Malignant','Benign']))

In [ ]:
# ─── 7. Model Comparison Bar Chart ───
comparison_df = pd.DataFrame({
    'Model':    list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'ROC-AUC':  [r['roc_auc']  for r in results.values()]
})

comparison_df.set_index('Model')[['Accuracy','ROC-AUC']].plot(
    kind='bar', figsize=(10, 5), edgecolor='white', colormap='viridis')
plt.title('Model Comparison — Disease Prediction')
plt.ylabel('Score')
plt.ylim(0.85, 1.0)
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('disease_model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print(comparison_df.to_string(index=False))

In [ ]:
# ─── 8. ROC Curve ───
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})")
plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Disease Prediction (Breast Cancer)')
plt.legend()
plt.tight_layout()
plt.savefig('disease_roc.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 9. Best Model — Confusion Matrix ───
best_name = max(results, key=lambda k: results[k]['roc_auc'])
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Malignant','Benign'],
            yticklabels=['Malignant','Benign'])
plt.title(f'Confusion Matrix — {best_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('disease_confusion.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Best Model: {best_name} | AUC = {results[best_name]["roc_auc"]:.4f}')

In [ ]:
# ─── 10. Feature Importance (Random Forest) ───
rf = results['Random Forest']['model']
fi = pd.Series(rf.feature_importances_, index=df_bc.drop('target', axis=1).columns)
fi.nlargest(15).sort_values().plot(kind='barh', figsize=(9, 6),
    color='steelblue', edgecolor='white')
plt.title('Top 15 Feature Importances — Random Forest (Breast Cancer)')
plt.tight_layout()
plt.savefig('disease_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 11. Predict for a New Patient ───
best_model = results[best_name]['model']

# Use first test sample as example
sample = X_test.iloc[[0]]
sample_scaled = scaler.transform(sample)
pred = best_model.predict(sample_scaled)[0]
prob = best_model.predict_proba(sample_scaled)[0]

label = 'Benign' if pred == 1 else 'Malignant'
print(f'Sample Prediction: {label}')
print(f'Malignant probability: {prob[0]:.4f}')
print(f'Benign probability   : {prob[1]:.4f}')

## ✅ Summary

| Model | Accuracy | ROC-AUC |
|---|---|---|
| Logistic Regression | — | — |
| SVM | — | — |
| Random Forest | — | — |
| XGBoost | — | — |

*(Fill after running)*

**Datasets used:** Breast Cancer (sklearn built-in), Heart Disease (UCI), Diabetes (Pima Indians)

**GitHub Repo:** `CodeAlpha_DiseasePrediction`